In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
from nemosis import static_table, dynamic_data_compiler, defaults
import plotly.express as px
import os
import glob
import dask.dataframe as dd
import re

raw_data_cache = '/Volumes/T7/Misc'

pd.set_option('display.max_columns', None)

In [6]:
# Gather all Parquet files in in a single dask data frame
all_files = glob.glob('/Volumes/T7/bid-volume-filtered-3/*.parquet')

# 2. Filter out any hidden dot-underscore files (._filename.parquet)
valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]

# 3. Print how many valid Parquet files were found
n_files = len(valid_files)
print(f"Found {n_files} valid Parquet file(s) in /Volumes/T7/bid-volume-filtered.\n")

# 4. Iterate over each valid file to show progress
for idx, file_path in enumerate(valid_files, start=1):
    file_name = os.path.basename(file_path)
    print(f"Processing file {idx} of {n_files}: {file_name}")

# 5. Read all valid files into a single Dask DataFrame
print("\nReading all valid Parquet files into a Dask DataFrame...")
ddf = dd.read_parquet(valid_files)
print("Done reading files into Dask DataFrame.\n")

# 7. Display columns and a small sample
print("Columns in the Dask DataFrame:", ddf.columns)
print("\nSample data from the Dask DataFrame:")
print(ddf.head())

Found 1448 valid Parquet file(s) in /Volumes/T7/bid-volume-filtered.

Processing file 1 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
Processing file 2 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
Processing file 3 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
Processing file 4 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20091001.parquet
Processing file 5 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20091101.parquet
Processing file 6 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20091201.parquet
Processing file 7 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100101.parquet
Processing file 8 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100201.parquet
Processing file 9 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100301.parquet
Processing file 10 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100401.parquet
Processing file 11 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100501.parquet
Processing file 12 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100601.parquet
Processing file 13 of 1448: PUBLIC_DVD_BIDPEROFFER_D_20100701.parquet
Processing file 14 of 1448: P

In [7]:
ddf.head()

,SETTLEMENTDATE,DUID,BIDTYPE,OFFERDATE,MAXAVAIL,ENABLEMENTMIN,ENABLEMENTMAX,LOWBREAKPOINT,HIGHBREAKPOINT,BANDAVAIL1,BANDAVAIL2,BANDAVAIL3,BANDAVAIL4,BANDAVAIL5,BANDAVAIL6,BANDAVAIL7,BANDAVAIL8,BANDAVAIL9,BANDAVAIL10,INTERVAL_DATETIME
0,2009/07/01 00:00:00,BASTYAN,LOWERREG,2009/07/01 15:01:02,26,25,78,51,78,0,0,0,88,0,0,0,0,0,0,2009/07/01 18:05:00
1,2009/07/01 00:00:00,BASTYAN,RAISEREG,2009/07/01 15:01:02,26,0,78,0,52,0,0,0,88,0,0,0,0,0,0,2009/07/01 18:05:00
2,2009/07/01 00:00:00,BELLBAY1,LOWERREG,2005/04/21 14:31:20,0,36,120,49,120,3,3,3,3,3,3,3,3,3,3,2009/07/01 18:05:00
3,2009/07/01 00:00:00,BELLBAY1,RAISEREG,2005/04/21 14:31:20,0,36,115,36,115,3,3,3,3,3,3,3,3,3,3,2009/07/01 18:05:00
4,2009/07/01 00:00:00,BELLBAY2,LOWERREG,2005/04/21 14:31:20,0,36,120,49,120,3,3,3,3,3,3,3,3,3,3,2009/07/01 18:05:00


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np
import calendar

def count_bids_per_day(input_dir, output_dir=None, exclude_dates=None):
    """
    Count bids per day from parquet files and create multiple time series plots
    with enhanced statistics and separate moving average plots
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files with bid data
    output_dir : str, optional
        Directory to save output files (defaults to input_dir if None)
    exclude_dates : list, optional
        List of dates to exclude from analysis (format: 'YYYY-MM-DD')
    """
    print(f"Counting bids per day from files in {input_dir}...")
    
    # Set default output directory
    if output_dir is None:
        output_dir = input_dir
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get list of all parquet files
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    file_list.sort()
    
    if not file_list:
        print(f"No parquet files found in {input_dir}")
        return
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Initialize a dictionary to store bid counts per day
    all_counts = {}
    bid_types = {}  # To track different bid types
    
    # Process each file
    for idx, file_path in enumerate(file_list, start=1):
        try:
            # Print progress every 10 files
            if idx % 10 == 0 or idx == 1 or idx == len(file_list):
                print(f"Processing file {idx}/{len(file_list)}: {os.path.basename(file_path)}")
            
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            if 'SETTLEMENTDATE' not in df.columns:
                print(f"  Warning: SETTLEMENTDATE column not found in {os.path.basename(file_path)}")
                continue
            
            # Extract the date part from SETTLEMENTDATE
            df['DATE'] = pd.to_datetime(df['SETTLEMENTDATE']).dt.date
            
            # Count bids per day
            daily_counts = df.groupby('DATE').size()
            
            # Update the overall counts
            for date, count in daily_counts.items():
                date_str = str(date)
                if date_str in all_counts:
                    all_counts[date_str] += count
                else:
                    all_counts[date_str] = count
            
            # Count by bid type if available
            if 'BIDTYPE' in df.columns:
                type_counts = df.groupby(['DATE', 'BIDTYPE']).size().reset_index(name='COUNT')
                for _, row in type_counts.iterrows():
                    date_str = str(row['DATE'])
                    bid_type = row['BIDTYPE']
                    count = row['COUNT']
                    
                    if date_str not in bid_types:
                        bid_types[date_str] = {}
                    
                    if bid_type in bid_types[date_str]:
                        bid_types[date_str][bid_type] += count
                    else:
                        bid_types[date_str][bid_type] = count
        
        except Exception as e:
            print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")
    
    if not all_counts:
        print("No bid data found in the files")
        return
    
    # Convert to DataFrame for easier handling
    dates = sorted(all_counts.keys())
    counts = [all_counts[date] for date in dates]
    
    count_df = pd.DataFrame({
        'date': dates,
        'total_bids': counts
    })
    
    # Convert date strings to datetime for proper sorting
    count_df['date'] = pd.to_datetime(count_df['date'])
    count_df = count_df.sort_values('date')
    
    # Add month and year columns for monthly aggregation
    count_df['month'] = count_df['date'].dt.month
    count_df['year'] = count_df['date'].dt.year
    count_df['month_name'] = count_df['date'].dt.month_name()
    
    # Add bid type columns if available
    if bid_types:
        # Find all unique bid types
        all_bid_types = set()
        for date_types in bid_types.values():
            all_bid_types.update(date_types.keys())
        
        # Add columns for each bid type
        for bid_type in all_bid_types:
            count_df[f'bids_{bid_type}'] = count_df['date'].astype(str).map(
                lambda date_str: bid_types.get(date_str, {}).get(bid_type, 0)
            )
    
    # Save original dataframe to CSV
    csv_path = os.path.join(output_dir, "fcas_bids_per_day_all_data.csv")
    count_df.to_csv(csv_path, index=False)
    print(f"Saved complete bid counts to {csv_path}")
    
    # Create plots and calculate statistics
    create_plots_and_stats(count_df, output_dir)
    
    # Create filtered version without excluded dates
    if exclude_dates:
        filtered_df = count_df.copy()
        for date_str in exclude_dates:
            filtered_df = filtered_df[filtered_df['date'] != pd.to_datetime(date_str)]
        
        # Save filtered dataframe
        filtered_csv_path = os.path.join(output_dir, "fcas_bids_per_day_filtered.csv")
        filtered_df.to_csv(filtered_csv_path, index=False)
        print(f"Saved filtered bid counts to {filtered_csv_path}")
        
        # Create plots for filtered data
        create_plots_and_stats(filtered_df, output_dir, suffix="_filtered")
    
    return count_df

def create_plots_and_stats(count_df, output_dir, suffix=""):
    """
    Create various plots and calculate enhanced statistics
    
    Parameters:
    -----------
    count_df : pandas DataFrame
        DataFrame containing bid counts
    output_dir : str
        Directory to save plots
    suffix : str, optional
        Suffix to append to filenames (e.g., "_filtered")
    """
    # Ensure we have the necessary bid type columns
    if not ('bids_RAISEREG' in count_df.columns and 'bids_LOWERREG' in count_df.columns):
        print("Warning: Missing RAISEREG or LOWERREG columns for separate plots")
        return
    
    # Set seaborn style for all plots
    sns.set_style("whitegrid")
    
    # Create a plots directory
    plots_dir = os.path.join(output_dir, f"plots{suffix}")
    os.makedirs(plots_dir, exist_ok=True)
    
    # List of columns to analyze
    columns = ['total_bids', 'bids_RAISEREG', 'bids_LOWERREG']
    
    # Calculate moving averages and statistics
    for col in columns:
        if col in count_df.columns:
            # Calculate statistics
            calculate_statistics(count_df, col, output_dir, suffix)
            
            # Generate all plots for this column
            create_plot_series(count_df, col, plots_dir, suffix)

def calculate_statistics(count_df, column, output_dir, suffix=""):
    """
    Calculate and print detailed statistics for a specific column
    """
    if column not in count_df.columns:
        return
    
    # Basic statistics
    max_val = count_df[column].max()
    max_date = count_df.loc[count_df[column].idxmax(), 'date'].date()
    min_val = count_df[column].min()
    min_date = count_df.loc[count_df[column].idxmin(), 'date'].date()
    avg_val = count_df[column].mean()
    std_val = count_df[column].std()
    median_val = count_df[column].median()
    
    # Format the column name for display
    col_name = column.replace('bids_', '')
    
    # Print statistics
    print(f"\n--- {col_name} Statistics{suffix} ---")
    print(f"Maximum: {max_val} on {max_date}")
    print(f"Minimum: {min_val} on {min_date}")
    print(f"Average: {avg_val:.1f}")
    print(f"Standard Deviation: {std_val:.1f}")
    print(f"Median: {median_val:.1f}")
    print(f"Coefficient of Variation: {(std_val/avg_val)*100:.1f}%")
    
    # Save statistics to a text file
    stats_file = os.path.join(output_dir, f"{col_name}_statistics{suffix}.txt")
    with open(stats_file, 'w') as f:
        f.write(f"--- {col_name} Statistics{suffix} ---\n")
        f.write(f"Date Range: {count_df['date'].min().date()} to {count_df['date'].max().date()}\n")
        f.write(f"Number of Days: {len(count_df)}\n\n")
        f.write(f"Maximum: {max_val} on {max_date}\n")
        f.write(f"Minimum: {min_val} on {min_date}\n")
        f.write(f"Average: {avg_val:.1f}\n")
        f.write(f"Standard Deviation: {std_val:.1f}\n")
        f.write(f"Median: {median_val:.1f}\n")
        f.write(f"Coefficient of Variation: {(std_val/avg_val)*100:.1f}%\n")
    
    print(f"Saved {col_name} statistics to {stats_file}")
    
    # Calculate monthly averages for additional insight
    monthly_stats = count_df.groupby(['year', 'month', 'month_name'])[column].agg(['mean', 'std', 'count']).reset_index()
    monthly_stats = monthly_stats.sort_values(['year', 'month'])
    
    # Save monthly statistics to CSV
    monthly_stats_file = os.path.join(output_dir, f"{col_name}_monthly_stats{suffix}.csv")
    monthly_stats.to_csv(monthly_stats_file, index=False)
    print(f"Saved {col_name} monthly statistics to {monthly_stats_file}")

def create_plot_series(count_df, column, plots_dir, suffix=""):
    """
    Create a series of plots for a specific column
    
    Parameters:
    -----------
    count_df : pandas DataFrame
        DataFrame containing bid data
    column : str
        Column name to plot
    plots_dir : str
        Directory to save plots
    suffix : str
        Suffix to append to filenames
    """
    col_name = column.replace('bids_', '')
    
    # 1. Original data plot (no moving average)
    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=count_df,
        x='date',
        y=column,
        linewidth=2,
        marker='o',
        markersize=3,
        alpha=0.7
    )
    
    plt.title(f'{col_name} per Day')
    plt.xlabel('Date')
    plt.ylabel(f'Number of {col_name}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Save original data plot
    original_plot_path = os.path.join(plots_dir, f"{col_name}_original{suffix}.png")
    plt.savefig(original_plot_path, dpi=300)
    plt.close()
    print(f"Saved original {col_name} plot to {original_plot_path}")
    
    # 2. Weekly (7-day) moving average plot
    if len(count_df) > 7:
        plt.figure(figsize=(12, 6))
        
        # Calculate 7-day moving average
        weekly_avg = count_df[column].rolling(window=7, center=True).mean()
        
        # Plot just the moving average
        plt.plot(count_df['date'], weekly_avg, 'b-', linewidth=2.5, label='7-day Moving Average')
        
        plt.title(f'{col_name} - 7-day Moving Average')
        plt.xlabel('Date')
        plt.ylabel(f'Number of {col_name}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Save weekly moving average plot
        weekly_plot_path = os.path.join(plots_dir, f"{col_name}_7day_avg{suffix}.png")
        plt.savefig(weekly_plot_path, dpi=300)
        plt.close()
        print(f"Saved 7-day moving average plot to {weekly_plot_path}")
    
    # 3. Monthly moving average plot
    if len(count_df) > 30:
        plt.figure(figsize=(12, 6))
        
        # Calculate 30-day moving average (approximately monthly)
        monthly_avg = count_df[column].rolling(window=30, center=True).mean()
        
        # Plot just the moving average
        plt.plot(count_df['date'], monthly_avg, 'g-', linewidth=2.5, label='30-day Moving Average')
        
        plt.title(f'{col_name} - 30-day Moving Average')
        plt.xlabel('Date')
        plt.ylabel(f'Number of {col_name}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Save monthly moving average plot
        monthly_plot_path = os.path.join(plots_dir, f"{col_name}_30day_avg{suffix}.png")
        plt.savefig(monthly_plot_path, dpi=300)
        plt.close()
        print(f"Saved 30-day moving average plot to {monthly_plot_path}")
    
    # 4. Comparison plot with original data and both moving averages
    plt.figure(figsize=(12, 6))
    
    # Plot the original data and both moving averages
    plt.plot(count_df['date'], count_df[column], 'b-', linewidth=1, marker='o', 
             markersize=2, alpha=0.4, label='Daily Data')
    
    if len(count_df) > 7:
        weekly_avg = count_df[column].rolling(window=7, center=True).mean()
        plt.plot(count_df['date'], weekly_avg, 'r-', linewidth=2, label='7-day Moving Average')
    
    if len(count_df) > 30:
        monthly_avg = count_df[column].rolling(window=30, center=True).mean()
        plt.plot(count_df['date'], monthly_avg, 'g-', linewidth=2, label='30-day Moving Average')
    
    plt.title(f'{col_name} - Data with Moving Averages')
    plt.xlabel('Date')
    plt.ylabel(f'Number of {col_name}')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    
    # Save comparison plot
    comparison_plot_path = os.path.join(plots_dir, f"{col_name}_comparison{suffix}.png")
    plt.savefig(comparison_plot_path, dpi=300)
    plt.close()
    print(f"Saved comparison plot to {comparison_plot_path}")
    
    # 5. Monthly aggregated bar chart
    plt.figure(figsize=(14, 8))
    
    # Group by year and month
    monthly_data = count_df.groupby(['year', 'month', 'month_name'])[column].mean().reset_index()
    monthly_data['month_year'] = monthly_data.apply(
        lambda x: f"{x['month_name'][:3]} {x['year']}", axis=1
    )
    monthly_data = monthly_data.sort_values(['year', 'month'])
    
    # Plot the monthly averages
    sns.barplot(data=monthly_data, x='month_year', y=column)
    
    plt.title(f'Average Daily {col_name} by Month')
    plt.xlabel('Month')
    plt.ylabel(f'Average Daily {col_name}')
    plt.xticks(rotation=90)
    plt.tight_layout()
    
    # Save monthly bar chart
    monthly_bar_path = os.path.join(plots_dir, f"{col_name}_monthly_avg{suffix}.png")
    plt.savefig(monthly_bar_path, dpi=300)
    plt.close()
    print(f"Saved monthly bar chart to {monthly_bar_path}")

if __name__ == "__main__":
    # Directory containing the parquet files
    input_directory = "/Volumes/T7/bid-volume-filtered-3"  # Update to your directory
    output_directory = "/Volumes/T7/bid-analysis"  # Where to save plots and CSV
    
    # List of dates to exclude from the filtered analysis
    exclude_dates = ["2016-09-06"]  # Add more dates if needed
    
    # Run the analysis
    bid_counts = count_bids_per_day(
        input_dir=input_directory,
        output_dir=output_directory,
        exclude_dates=exclude_dates
    )

Saved improved monthly chart to /Volumes/T7/bid-analysis/plots/total_bids_monthly_chart.png
Saved improved monthly chart to /Volumes/T7/bid-analysis/plots/bids_RAISEREG_monthly_chart.png
Saved improved monthly chart to /Volumes/T7/bid-analysis/plots/bids_LOWERREG_monthly_chart.png
All improved charts created successfully!


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np
import calendar
from matplotlib.dates import MonthLocator, DateFormatter

def create_improved_monthly_chart(count_df, output_dir, column='total_bids', suffix=""):
    """
    Create an improved monthly bar chart with better formatting
    
    Parameters:
    -----------
    count_df : pandas DataFrame
        DataFrame containing bid data
    output_dir : str
        Directory to save plots
    column : str
        Column name to plot (default: 'total_bids')
    suffix : str
        Suffix to append to filenames
    """
    # Create a plots directory if it doesn't exist
    plots_dir = os.path.join(output_dir, f"plots{suffix}")
    os.makedirs(plots_dir, exist_ok=True)
    
    # Use a cleaner column name for display (remove underscore)
    col_display_name = column.replace('_', ' ')
    if column.startswith('bids_'):
        col_display_name = column.replace('bids_', '')
    
    # Prepare data
    # Group by year and month, then calculate mean
    monthly_data = count_df.groupby(['year', 'month'])[column].mean().reset_index()
    
    # Create date objects for proper sorting and formatting
    monthly_data['date'] = pd.to_datetime(monthly_data.apply(
        lambda x: f"{int(x['year'])}-{int(x['month'])}-01", axis=1
    ))
    
    # Sort by date
    monthly_data = monthly_data.sort_values('date')
    
    # Create the plot
    plt.figure(figsize=(14, 8))
    
    # Use a blue color palette with different shades
    if len(monthly_data) > 10:
        # For many months, use a color gradient
        blues = plt.cm.Blues(np.linspace(0.4, 1.0, len(monthly_data)))
    else:
        # For fewer months, use a discrete palette
        blues = sns.color_palette("Blues", len(monthly_data))
    
    # Create bars with blue colormap
    bars = plt.bar(monthly_data['date'], monthly_data[column], width=20, color=blues)
    
    # Format x-axis to show only March of each year
    ax = plt.gca()
    
    # Get min and max years
    min_year = monthly_data['year'].min()
    max_year = monthly_data['year'].max()
    
    # Create custom tick positions and labels (March of each year)
    tick_dates = []
    tick_labels = []
    
    for year in range(min_year, max_year + 1):
        # Use March of each year as the tick
        march_date = pd.Timestamp(f"{year}-03-01")
        if march_date >= monthly_data['date'].min() and march_date <= monthly_data['date'].max():
            tick_dates.append(march_date)
            tick_labels.append(f"Mar {year}")
    
    # Set the custom ticks
    plt.xticks(tick_dates, tick_labels, rotation=45)
    
    # Add title and labels with properly formatted column name
    plt.title(f'Average Daily {col_display_name.title()} by Month', fontsize=14)
    plt.xlabel('Month', fontsize=12)
    plt.ylabel(f'Average Daily {col_display_name.title()}', fontsize=12)
    
    # Add grid for readability
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    # Save the improved chart
    chart_path = os.path.join(plots_dir, f"{column}_monthly_chart{suffix}.png")
    plt.savefig(chart_path, dpi=300)
    plt.close()
    
    print(f"Saved improved monthly chart to {chart_path}")
    
    return chart_path

if __name__ == "__main__":
    # Example usage (assuming you have the data loaded)
    # This would typically be called from your main analysis script
    
    # Load your existing CSV data
    output_directory = "/Volumes/T7/bid-analysis"  # Update to your directory
    data_file = os.path.join(output_directory, "fcas_bids_per_day_all_data.csv")
    
    if os.path.exists(data_file):
        # Load the data
        count_df = pd.read_csv(data_file)
        
        # Convert date to datetime
        count_df['date'] = pd.to_datetime(count_df['date'])
        
        # Add month and year columns if they don't exist
        if 'month' not in count_df.columns:
            count_df['month'] = count_df['date'].dt.month
        if 'year' not in count_df.columns:
            count_df['year'] = count_df['date'].dt.year
        
        # Create the improved charts
        create_improved_monthly_chart(count_df, output_directory, column='total_bids')
        
        # Also create charts for RAISEREG and LOWERREG if available
        if 'bids_RAISEREG' in count_df.columns:
            create_improved_monthly_chart(count_df, output_directory, column='bids_RAISEREG')
        if 'bids_LOWERREG' in count_df.columns:
            create_improved_monthly_chart(count_df, output_directory, column='bids_LOWERREG')
        
        print("All improved charts created successfully!")
    else:
        print(f"Data file not found: {data_file}")
        print("Please run the main analysis script first to generate the data file.")